# Suite CONF — Conformance across implementations

Every implementation writes the same files for the conformance corpus (`conformance/`): the certificates of one
fixture's run, suite RRN's (a session over a shelf of `bolt` and `nut`, each candidate of `Label` applied with the
caller's choice and certified), as mbse-schemas' JSON snapshot of the certificates' store
(`tests/conformance.ts`). This suite checks that this implementation's files are current, and that they are
byte-identical to every other implementation's: the certificates are the same data on the wire in either language.

In [ ]:
import { readdirSync, readFileSync, statSync } from "node:fs";
import { join } from "node:path";

import { render, ROOT } from "./conformance.js";

import { assert, equal } from "./support.js";

const MINE = join(ROOT, "typescript5");
const OTHERS = readdirSync(ROOT).sort().filter((d) => statSync(join(ROOT, d)).isDirectory() && d !== "typescript5")
  .map((d) => join(ROOT, d));
const files = render();
const read = (path: string) => readFileSync(path, "utf8");

<a id="CONF-01"></a>
## CONF-01 · This implementation's files are current

In [ ]:
assert(equal(readdirSync(MINE).sort(), [...files.keys()].sort()));
for (const [name, text] of files) assert(read(join(MINE, name)) === text, `${name} is stale: run npm run conformance`);
console.log([...files.keys()].sort());

<a id="CONF-02"></a>
## CONF-02 · Every other implementation wrote the same bytes

In [ ]:
assert(equal(OTHERS.map((d) => d.split("/").at(-1)), ["python3"]));
for (const other of OTHERS) {
  assert(equal(readdirSync(other).sort(), [...files.keys()].sort()));
  for (const [name, text] of files) assert(read(join(other, name)) === text, `${other}/${name} differs`);
}
console.log("identical to", OTHERS.map((d) => d.split("/").at(-1)));